In [2]:
import pandas as pd
from pathlib import Path

In [41]:
repo = Path(".").resolve().parent
json_path = repo / "data/raw/json"
print(json_path)

/home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/json


In [42]:
from json_parser_theo import load_chats_from_folder
df = load_chats_from_folder(json_path)
len(df.groupby("chat_id"))

30

In [43]:
df

,chat_id,file_name,turn,role,content
0,1,chat_export,1,user,wie codiere ich alle values in einen pd df zu ...
1,1,chat_export,2,assistant,In pandas kannst du alle negativen Werte im Da...
2,1,chat_export,3,user,Exportiere mir den chat bis hierher als json D...
3,2,copilot,1,user,Was sind wichtige Werke in der Literatur zur r...
4,2,copilot,2,assistant,**Kurzfazit:** \nDie Literatur zur **Reliabil...
...,...,...,...,...,...
284,30,role user content was2,1,user,was ist der unteschied zwischen computer visio...
285,30,role user content was2,2,assistant,"Der zentrale Unterschied liegt darin, **wie Mo..."
286,30,role user content was2,3,user,i need literatur on how to evaluate zero shot ...
287,30,role user content was2,4,assistant,If you're looking specifically for **how to ev...


In [44]:

anzahl_dateien = len([f for f in json_path.iterdir() if f.is_file()])

print(anzahl_dateien)

30


In [46]:
#df.groupby("chat_id")["turn"].max().eq(1)

In [47]:
df_user = df[df["role"] == "user"]
df_user

,chat_id,file_name,turn,role,content
0,1,chat_export,1,user,wie codiere ich alle values in einen pd df zu ...
2,1,chat_export,3,user,Exportiere mir den chat bis hierher als json D...
3,2,copilot,1,user,Was sind wichtige Werke in der Literatur zur r...
6,2,copilot,4,user,Exportiere den gesamten im aktuellen Kontext e...
8,2,copilot,6,user,Exportiere den gesamten sichtbaren Chatverlauf...
...,...,...,...,...,...
281,29,role user content was,1,user,was wären motivationen für die replikation die...
283,29,role user content was,3,user,Gebe mir den gesamten sichtbaren bisherigen Di...
284,30,role user content was2,1,user,was ist der unteschied zwischen computer visio...
286,30,role user content was2,3,user,i need literatur on how to evaluate zero shot ...


In [109]:
#shuffel dataset and reset ids
rng = np.random.default_rng(seed=42)
unique_ids = df_user['chat_id'].unique()
shuffled_ids = rng.permutation(unique_ids)

id_mapping = {old_id: new_id for new_id, old_id in enumerate(shuffled_ids, start=1)}

df_shuffled = (
    pd.concat([df_user[df_user['chat_id'] == i] for i in shuffled_ids])
    .reset_index(drop=True)
)

df_shuffled['chat_id'] = df_shuffled['chat_id'].map(id_mapping)

df_shuffled


,chat_id,file_name,turn,role,content
0,1,role user content was2,1,user,was ist der unteschied zwischen computer visio...
1,1,role user content was2,3,user,i need literatur on how to evaluate zero shot ...
2,1,role user content was2,5,user,Gebe mir den gesamten sichtbaren bisherigen Di...
3,2,laura-06,1,user,Version control conflict markers detected. Ple...
4,2,laura-06,2,user,what if I don't want to keep any of my changes...
...,...,...,...,...,...
166,30,gemini7,6,user,Exportiere den gesamten im aktuellen Kontext e...
167,30,gemini7,7,user,Schreibe einen Text für meine Hausarbeit (Repl...
168,30,gemini7,9,user,Alles in einem Fließtext ohne Überschriften bi...
169,30,gemini7,11,user,Wie nennt man es wenn man einer studie komplet...


In [48]:

df_control = pd.DataFrame({
    "chat_id": range(1, 31),
    "task": ""*30,
    "sentiment": ""*30,
    "critical": ""*30
})

In [ ]:
df_control

In [110]:
df_shuffled.to_csv(repo / "data/processed/control/user_shuffel.csv", index=False)
df_control.to_csv(repo / "data/processed/control/control.csv", index=False)
df_user.to_csv(repo / "data/processed/control/user.csv", index=False)

In [32]:
!pip install krippendorff

In [49]:
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
import krippendorff

In [87]:
control_a = pd.read_csv(repo / "data/processed/control/control_hannah.csv", delimiter = ";")
control_b = pd.read_csv(repo / "data/processed/control/control_theo_2.csv")
control_c = pd.read_csv(repo / "data/processed/control/control_laura.csv")


In [99]:
def kippendorff_alpha(name: str,
                      annontation = ["code_coder_A", "code_coder_C",  "code_coder_B"],
                      messumerment = "nominal"
                        ):
    coder_a = control_a[["chat_id", name]].rename(columns={name: "code_coder_A"})
    coder_b = control_b[["chat_id", name]].rename(columns={name: "code_coder_B"})
    coder_c = control_c[["chat_id", name]].rename(columns={name: "code_coder_C"})
    merged = df_user.merge(coder_a, on="chat_id").merge(coder_b, on="chat_id").merge(coder_c, on="chat_id")
    alpha = krippendorff.alpha(
    reliability_data=merged[annontation].T.to_numpy(),
    level_of_measurement=messumerment
    )
    return alpha


In [102]:
kippendorff_alpha(name = "sentiment", annontation = ["code_coder_A", "code_coder_C",  "code_coder_B"], messumerment = "nominal")

np.float64(0.04274934983256584)

In [105]:
coderABC = ["code_coder_A", "code_coder_C", "code_coder_B"]
coder_AB = ["code_coder_A", "code_coder_B"]
coder_BC = ["code_coder_B", "code_coder_C"]
coder_AC = ["code_coder_A", "code_coder_C"]


for coder in  [coderABC,coder_AB, coder_BC,coder_AC]:
    print("")
    for name, messure in zip(["task", "sentiment", "critical"], ["nominal", "ordinal", "ordinal"]):
        alpha = kippendorff_alpha(name, annontation = coder, messumerment = messure)
        print(f"Annotators : {coder};" f"Krippendorff's alpha for {name}: {alpha:.4f}")


Annotators : ['code_coder_A', 'code_coder_C', 'code_coder_B'];Krippendorff's alpha for task: 0.6275
Annotators : ['code_coder_A', 'code_coder_C', 'code_coder_B'];Krippendorff's alpha for sentiment: 0.0427
Annotators : ['code_coder_A', 'code_coder_C', 'code_coder_B'];Krippendorff's alpha for critical: 0.3589

Annotators : ['code_coder_A', 'code_coder_B'];Krippendorff's alpha for task: 0.8376
Annotators : ['code_coder_A', 'code_coder_B'];Krippendorff's alpha for sentiment: 0.7278
Annotators : ['code_coder_A', 'code_coder_B'];Krippendorff's alpha for critical: 0.8925

Annotators : ['code_coder_B', 'code_coder_C'];Krippendorff's alpha for task: 0.4494
Annotators : ['code_coder_B', 'code_coder_C'];Krippendorff's alpha for sentiment: -0.3954
Annotators : ['code_coder_B', 'code_coder_C'];Krippendorff's alpha for critical: -0.0629

Annotators : ['code_coder_A', 'code_coder_C'];Krippendorff's alpha for task: 0.6068
Annotators : ['code_coder_A', 'code_coder_C'];Krippendorff's alpha for sentimen

In [77]:
def cohans_kappa(name: str,
                      annontation = ["code_coder_A", "code_coder_C",  "code_coder_B"],
                        ):
    coder_a = control_a[["chat_id", name]].rename(columns={name: "code_coder_A"})
    coder_b = control_b[["chat_id", name]].rename(columns={name: "code_coder_B"})
    coder_b Coder
    coder_c = control_c[["chat_id", name]].rename(columns={name: "code_coder_C"})
    merged = df_user.merge(coder_a, on="chat_id").merge(coder_b, on="chat_id").merge(coder_c, on="chat_id")
    kappa = cohen_kappa_score(merged[annontation[0]], merged[annontation[1]])
    return kappa

In [91]:
print(cohans_kappa(name = "task", annontation = coder_Aexport OPENAI_API_KEY="dein_api_key"C))
print(cohans_kappa(name = "sentiment", annontation = coder_AC))
print(cohans_kappa(name = "critical", annontation = coder_AC))

0.6168316831683169
0.012389380530973493
0.20274088725628703
